In [3]:
class CollinearityRemover(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.features_to_keep_ = None
        
    def fit(self, X, y=None):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > self.threshold)]
        self.features_to_keep_ = [i for i in range(X.shape[1]) if i not in to_drop]
        return self
    
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.iloc[:, self.features_to_keep_]
        return X[:, self.features_to_keep_]

In [49]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import (
    SelectKBest, 
    f_regression, 
    VarianceThreshold,
    SelectFromModel,
    RFE, 
    RFECV
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_log_error

from sklearn import set_config
set_config(transform_output='pandas')



data = pd.read_csv(r"C:\Users\User\Downloads\5_housing_iteration_6_regression\housing_iteration_6_regression\housing_iteration_6_regression.csv")

data1 = data.copy()

data1=data1.set_index("Id")

X = data1 
y = X.pop("SalePrice")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)


# train_data = X_train.copy()
# train_data["SalePrice"] = y_train

X_num = X_train.select_dtypes(include='number').columns.tolist()
X_cat = X_train.select_dtypes(exclude='number').columns.tolist()

ordinal_features = [
    'OverallQual','OverallCond','ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual',
    'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]

ordinal_categories = [
    ['1','2','3','4','5','6','7','8','9','10'],         # OverallQual
    ['1','2','3','4','5','6','7','8','9','10'],         # OverallCond
    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],                     # ExterQual
    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],                     # ExterCond
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],               # BsmtQual
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],               # BsmtCond
    ['NA', 'No', 'Mn', 'Av', 'Gd'],                     # BsmtExposure
    ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],  # BsmtFinType1
    ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],  # BsmtFinType2
    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],                     # HeatingQC
    ['Po', 'Fa', 'TA', 'Gd', 'Ex'],                     # KitchenQual
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],               # FireplaceQu
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],               # GarageQual
    ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],               # GarageCond
    ['NA', 'Fa', 'TA', 'Gd', 'Ex']                      # PoolQC
]

oneHot_feat = list(set(X_cat) - set(ordinal_features))

# ── Pipelines per branch ───────────────────────────────────────────────────────
num_pipeline = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler()
)

ordinal_pipeline = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='NA'),
    OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )
)

onehot_pipeline = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='NA'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

preprocessor = ColumnTransformer([
    ('num',     num_pipeline,     X_num),
    ('ordinal', ordinal_pipeline, ordinal_features),
    ('onehot',  onehot_pipeline,  oneHot_feat)
])


In [50]:
knn_kbest_pipeline = make_pipeline(
    preprocessor,
    VarianceThreshold(),
    CollinearityRemover(),
    SelectKBest(score_func=f_regression),
    KNeighborsRegressor()
)

param_grid_knn_kbest = {
    "variancethreshold__threshold": [0.0],            # fixed — 0.01 always worse
    "collinearityremover__threshold": [0.95],          # fixed — 0.90 too aggressive
    "selectkbest__k": [24, 25, 26, 27, 28],            # fine-search around 26
    "kneighborsregressor__n_neighbors": [8, 9, 10, 11], # fine-search around 8
    "kneighborsregressor__weights": ["distance"],      # uniform always worse
    "kneighborsregressor__p": [1]                   # 1 (Manhattan) usually wins
}

grid_knn_kbest = GridSearchCV(
    knn_kbest_pipeline,
    param_grid_knn_kbest,
    cv=5,
    scoring='neg_root_mean_squared_log_error',
    n_jobs=-1,
    verbose=1
)

grid_knn_kbest.fit(X_train, y_train)
print(f"\nBest RMSLE: {-grid_knn_kbest.best_score_:.4f}")
print(f"Best params: {grid_knn_kbest.best_params_}")


y_pred = grid_knn_kbest.predict(X_test)
y_pred = np.clip(y_pred, 1, None)

print(f"\nTest  RMSLE   : {root_mean_squared_log_error(y_test, y_pred):.4f}")
print(f"Test  MAE     : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Test  R²      : {r2_score(y_test, y_pred):.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best RMSLE: 0.1667
Best params: {'collinearityremover__threshold': 0.95, 'kneighborsregressor__n_neighbors': 11, 'kneighborsregressor__p': 1, 'kneighborsregressor__weights': 'distance', 'selectkbest__k': 26, 'variancethreshold__threshold': 0.0}

Test  RMSLE   : 0.1432
Test  MAE     : 18606.18
Test  R²      : 0.8424


In [51]:
grid_knn_kbest.fit(X, y)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'collinearityremover__threshold': [0.95], 'kneighborsregressor__n_neighbors': [8, 9, ...], 'kneighborsregressor__p': [1], 'kneighborsregressor__weights': ['distance'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_log_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the hi

In [52]:
print(f"\nBest RMSLE: {-grid_knn_kbest.best_score_:.4f}")
print(f"Best params: {grid_knn_kbest.best_params_}")

y_pred = grid_knn_kbest.predict(X)
y_pred = np.clip(y_pred, 1, None)

print(f"\nTest  RMSLE   : {root_mean_squared_log_error(y, y_pred):.4f}")
print(f"Test  MAE     : {mean_absolute_error(y, y_pred):.2f}")
print(f"Test  R²      : {r2_score(y, y_pred):.4f}")


Best RMSLE: 0.1615
Best params: {'collinearityremover__threshold': 0.95, 'kneighborsregressor__n_neighbors': 9, 'kneighborsregressor__p': 1, 'kneighborsregressor__weights': 'distance', 'selectkbest__k': 25, 'variancethreshold__threshold': 0.0}

Test  RMSLE   : 0.0022
Test  MAE     : 20.66
Test  R²      : 1.0000


In [42]:
test_data = pd.read_csv(r"C:\Users\User\Downloads\regression\test.csv")
test_data = test_data.set_index('Id')


In [43]:
test_data.shape

(1459, 79)

In [44]:
test_data.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [45]:
test_data['SalePrice'] = grid_knn_kbest.predict(test_data)

In [23]:
test_data['SalePrice'].to_csv(r"C:\Users\User\Downloads\regression\submission.csv")